<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase35_Generalization_Hybrid_Carrier_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 35 — Person-disjoint generalization and mixed-study adaptive carrier
**Google Drive → Google Drive · synthetic-only · runs without Athena**

**Question:** Does the Phase 34 metadata-size result survive a prespecified *person-disjoint* synthetic holdout and a stress case containing two different MRI studies in the same visit? Compare Phase 34's five frozen plan definitions with an explicitly new candidate `HYBRID_CODED` that shares visit context only when unambiguous and otherwise falls back to the study level *for the affected visit*, rather than rejecting the entire data slice. Include all dictionary and envelope bytes in each representation's reported metadata size.

**Research integrity:** This is an algorithmic transport experiment, not proof of novelty or clinical efficacy. The entire source fixture is developer-generated Phase 33 synthetic data; its 9-person holdout is not externally independent. A codebook learned from the records being encoded is allowed, but the plan family is selected **using only the 10 training people's source metadata**, without injected fault labels, target test outcomes, or clinical diagnoses. Phase 34's 38.98% was achieved on the whole original synthetic dataset; do not represent it as a held-out estimate. The synthetic proxy cohort is *not* an MCI cohort. No Athena concepts are invented, no OMOP/MI-CDM load or real MRI is performed, and original generic TFL Arm C remains frozen/nonportable.

**Input requirement:** Keep all eight files written by the original Phase 34 Colab in the **same Google Drive folder**. The notebook independently verifies each input artifact against its hash in `ICHI2027_Phase34_SHAREABLE.json`. If you only downloaded four Phase 34 outputs, rerun Phase 34 in Colab to retain its complete Drive output set; you do *not* need to rerun earlier phases.

In [1]:
import os
DRIVE_ROOT_OVERRIDE=os.environ.get('ICHI2027_DRIVE_ROOT','').strip() # optional local test only
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime,timezone
import os,re,csv,json,copy,hashlib,math
ROOT=Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
PREFERRED_PHASE34_DIR=''  # Optional: exact folder containing your Phase34 files
EXPECTED_PHASE='34_adaptive_synthetic_evidence_carrier_prototype'
INPUTS={
 'source':'ICHI2027_Phase34_CANONICAL_SOURCE_MANIFEST.csv',
 'base':'ICHI2027_Phase34_VALUE_ONLY_SHADOW.csv',
 'carrier':'ICHI2027_Phase34_ADAPTIVE_EVIDENCE_CARRIER.json',
 'plans':'ICHI2027_Phase34_ENCODING_PLAN_AUDIT.csv',
 'tests':'ICHI2027_Phase34_RECONSTRUCTION_TESTS.csv',
 'cohort':'ICHI2027_Phase34_SOURCE_RESTORED_PROXY_COMPARISON.csv',
 'blockers':'ICHI2027_Phase34_RESEARCH_GATES.csv',
}
def sha_bytes(bb):return hashlib.sha256(bb).hexdigest()
def sha(path):
 h=hashlib.sha256()
 with path.open('rb') as f:
    for bb in iter(lambda:f.read(1024*1024),b''):h.update(bb)
 return h.hexdigest()
def canonical(obj):return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def byte_length(obj):return len(canonical(obj).encode('utf-8'))
def load_csv(path):
 with path.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def files_safe(folder):
 for root,dirs,files in os.walk(str(folder),onerror=lambda _:None):
    dirs[:]=[d for d in dirs if d not in ['.git','_PRIVATE_ATHENA','.ipynb_checkpoints']]
    for name in sorted(files):yield Path(root)/name
def name_matches(real,base):
 stem,ext=os.path.splitext(base)
 return real==base or bool(re.fullmatch(re.escape(stem)+r'\(\d+\)'+re.escape(ext),real))
def file_with_hash(folder,base,expected):
 try:
    names=sorted(folder.iterdir(),key=lambda x:x.name)
 except (OSError,FileNotFoundError):return None
 for p in names:
    if not name_matches(p.name,base) or not p.is_file():continue
    try:
       if sha(p)==expected:return p
    except (OSError,FileNotFoundError):continue
 return None
assert ROOT.is_dir(),'Google Drive root not mounted: '+str(ROOT)
search_root=Path(PREFERRED_PHASE34_DIR) if PREFERRED_PHASE34_DIR else ROOT
candidates=[p for p in files_safe(search_root) if name_matches(p.name,'ICHI2027_Phase34_SHAREABLE.json')]
SELECTED=None; WHY=[]
for p in candidates:
 try:
    info=json.loads(p.read_text(encoding='utf-8'))
    if info.get('phase')!=EXPECTED_PHASE:continue
    found={k:file_with_hash(p.parent,name,info['artifact_sha256'][k]) for k,name in INPUTS.items()}
    if all(found.values()):SELECTED=(p,info,found);break
    WHY.append({'folder':str(p.parent),'missing_or_hash_mismatch':[k for k,x in found.items() if x is None]})
 except (OSError,ValueError,KeyError,FileNotFoundError) as ex:
    WHY.append({'summary':str(p),'error':str(ex)})
if SELECTED is None:
 raise FileNotFoundError('No full same-folder Phase34 run matched all seven declared artifact hashes. '
 'Check that CANONICAL_SOURCE_MANIFEST and ADAPTIVE_EVIDENCE_CARRIER JSON are still in Phase34 Drive folder. '
 'Do not copy older files under newer summary names. Candidates: '+str(WHY[:5]))
P34_PATH,P34,P34_FILES=SELECTED; OUT_DIR=P34_PATH.parent
assert P34['method']['status']=='ALGORITHMIC_PROTOTYPE_NOT_VALIDATED_NOVELTY'
assert P34['synthetic_fixture']['people']==19 and P34['synthetic_fixture']['observations']==148
assert P34['unresolved']['original_vocabulary_gaps_blocked']==36
assert P34['unresolved']['synthetic_faithful_armA_completed'] is False
assert P34['unresolved']['official_MiCDM_loading'] is False
assert P34['tests']['synthetic_transport_tests_passing']==9
assert P34['method']['selected_plan']=='VISIT_CODED'
SOURCE=load_csv(P34_FILES['source']);BASE34=load_csv(P34_FILES['base'])
for row in SOURCE:
 row['provenance_sources']=json.loads(row['provenance_sources'])
assert len(SOURCE)==148 and len(BASE34)==148
assert sha_bytes(canonical(sorted(SOURCE,key=lambda x:x['observation_id'])).encode('utf-8'))==P34['method']['full_source_selected_field_manifest_sha256'], 'Phase34 source canonical digest mismatch'
assert all(str(row.get('passed')).strip().lower()=='true' for row in load_csv(P34_FILES['tests']))
COHORT34=load_csv(P34_FILES['cohort']); assert len(COHORT34)==19
assert sum(x['source_proxy_member']=='True' for x in COHORT34)==15
print('Verified complete, matching Phase34 folder:',OUT_DIR)
print('Phase34 hash gate: PASS, 148 synthetic observations / 19 synthetic people.')

Verified complete, matching Phase34 folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
Phase34 hash gate: PASS, 148 synthetic observations / 19 synthetic people.


## 1 · Re-execute the *frozen Phase 34* five-plan codec
The next cell contains the Phase 34 encoder and decoder definitions **embedded verbatim** for reproducible comparison. It is not an alteration to that prior notebook and does not run the original Arm C. First reproduce the exact Phase 34 full-cohort canonical metadata byte count and source-field reconstruction, then evaluate unseen persons.

In [3]:
BASE_FIELDS=['observation_id','person_ref','visit','effective_datetime','value_raw']
GROUP_FIELDS=['study_ref','study_uid','series_uid','study_started','modality','device_ref']
LOCAL_FIELDS=['source_observation_ref','region','side','unit','trace_state','trace_rule','provenance_id','provenance_sources']
CONTEXT_FIELDS=GROUP_FIELDS+LOCAL_FIELDS
assert sorted(set(BASE_FIELDS+CONTEXT_FIELDS))==sorted(SOURCE[0].keys())
def context(r):return {k:copy.deepcopy(r[k]) for k in CONTEXT_FIELDS}
def base_table(source):return [{k:copy.deepcopy(r[k]) for k in BASE_FIELDS} for r in source]
def visit_key(row):return canonical([row['person_ref'],row['visit']])
class PlanError(ValueError):pass

def make_carrier(source,plan):
    if plan not in ['INLINE','STUDY','VISIT','STUDY_CODED','VISIT_CODED']:
        raise PlanError('Unknown plan')
    use_study=plan.startswith('STUDY')
    use_visit=plan.startswith('VISIT')
    coded=plan.endswith('_CODED')
    source=sorted(source,key=lambda x:x['observation_id'])
    if len({r['observation_id'] for r in source})!=len(source):raise PlanError('duplicate observation')
    payload={'format':'ICHI2027_PHASE34_ALPHA1','plan':plan,'row_context':{},'group_context':{},'codebooks':{}}
    groups={}
    if use_study or use_visit:
        for r in source:
            key=r['study_ref'] if use_study else visit_key(r)
            g={k:copy.deepcopy(r[k]) for k in GROUP_FIELDS}
            if key in groups and groups[key]!=g:
                raise PlanError('Unsafe inferred source functional dependency for '+key)
            groups[key]=g
        payload['group_context']={k:groups[k] for k in sorted(groups)}
    if coded:
        anatomy=sorted({canonical([r['region'],r['side']]) for r in source})
        traces=sorted({canonical([r['trace_state'],r['trace_rule']]) for r in source})
        a_index={key:f'A{i:03d}' for i,key in enumerate(anatomy)}
        t_index={key:f'T{i:03d}' for i,key in enumerate(traces)}
        units={r['unit'] for r in source}
        one_unit=next(iter(units)) if len(units)==1 else None
        payload['codebooks']={
            'anatomy':{a_index[k]:json.loads(k) for k in anatomy},
            'trace':{t_index[k]:json.loads(k) for k in traces},
            'global_unit':one_unit}
    for r in source:
        if plan=='INLINE':
            row_ctx=context(r)
        else:
            row_ctx={k:copy.deepcopy(r[k]) for k in LOCAL_FIELDS}
            if use_study:row_ctx['study_ref']=r['study_ref']
            if coded:
                row_ctx['anatomy_code']=a_index[canonical([r['region'],r['side']])]
                row_ctx['trace_code']=t_index[canonical([r['trace_state'],r['trace_rule']])]
                for key in ['region','side','trace_state','trace_rule']:del row_ctx[key]
                if one_unit is not None:del row_ctx['unit']
        payload['row_context'][r['observation_id']]=row_ctx
    return payload

def reconstruct(base,carrier):
    plan=carrier.get('plan','')
    if carrier.get('format')!='ICHI2027_PHASE34_ALPHA1':raise PlanError('Wrong carrier format')
    rows=carrier['row_context'];groups=carrier['group_context'];codes=carrier['codebooks']
    if set(rows)!={r['observation_id'] for r in base}:raise PlanError('Missing or extra row context')
    reconstructed=[]
    for b in base:
        rowctx=copy.deepcopy(rows[b['observation_id']]);r=copy.deepcopy(b)
        if plan=='INLINE':
            r.update(rowctx)
        else:
            if plan.endswith('_CODED'):
                try:
                    r['region'],r['side']=codes['anatomy'][rowctx.pop('anatomy_code')]
                    r['trace_state'],r['trace_rule']=codes['trace'][rowctx.pop('trace_code')]
                    if codes['global_unit'] is not None:r['unit']=codes['global_unit']
                except (KeyError,IndexError,TypeError) as exc:raise PlanError('Invalid/missing codebook entry') from exc
            if plan.startswith('STUDY'):
                key=rowctx['study_ref']
            elif plan.startswith('VISIT'):
                key=visit_key(b)
            else:raise PlanError('Unknown carrier plan')
            try:r.update(groups[key])
            except KeyError as exc:raise PlanError('Unknown group key') from exc
            r.update(rowctx)
        if set(r)!=set(BASE_FIELDS+CONTEXT_FIELDS):
            raise PlanError(f'Missing/unexpected evidence fields {b["observation_id"]}: {sorted(set(r)^set(BASE_FIELDS+CONTEXT_FIELDS))}')
        reconstructed.append(r)
    return sorted(reconstructed,key=lambda x:x['observation_id'])

In [4]:
assert len(SOURCE[0])==len(BASE_FIELDS+CONTEXT_FIELDS)==19
assert all(sorted(row)==sorted(BASE_FIELDS+CONTEXT_FIELDS) for row in SOURCE)
assert canonical(sorted(base_table(SOURCE),key=lambda x:x['observation_id']))==canonical(sorted(BASE34,key=lambda x:x['observation_id']))
C34=json.loads(P34_FILES['carrier'].read_text(encoding='utf-8'))
assert C34['plan']=='VISIT_CODED'
assert byte_length(C34)==P34['method']['selected_metadata_bytes']==56408
R34=reconstruct(base_table(SOURCE),C34)
assert canonical(R34)==canonical(sorted(SOURCE,key=lambda x:x['observation_id']))
assert byte_length(make_carrier(SOURCE,'INLINE'))==P34['method']['inline_metadata_bytes']==92435
print('Frozen Phase34 comparison reproduced: 148/148 selected-field roundtrip, 56,408 vs 92,435 metadata bytes.')

Frozen Phase34 comparison reproduced: 148/148 selected-field roundtrip, 56,408 vs 92,435 metadata bytes.


## 2 · Define a constrained hybrid for ambiguous visits
`HYBRID_CODED` stores common imaging context once per **person/visit** when unique. If multiple studies are present within a visit, it stores those study contexts in a separate dictionary, and **only those ambiguous rows** carry their study reference. Anatomy and TRACE codebooks are counted in metadata bytes. It is **source-field reversible only**; a source row that disagrees with its own study reference or provenance is quarantined *before* compression. No synthetic source is interpreted as a real procedure or DICOM record.

In [5]:
HYBRID='HYBRID_CODED'
CANDIDATES=['INLINE','STUDY','VISIT','STUDY_CODED','VISIT_CODED',HYBRID]
PH35_FORMAT='ICHI2027_PHASE35_HYBRID_BETA1'
def assert_source_integrity(source):
  if not source:raise PlanError('Empty source subset')
  if len({r['observation_id'] for r in source})!=len(source):raise PlanError('Repeated observation key')
  if len({r['source_observation_ref'] for r in source})!=len(source):raise PlanError('Repeated FHIR source observation')
  study_context={};study_person={};study_uids={};series_uids={}
  for r in source:
    if sorted(r)!=sorted(BASE_FIELDS+CONTEXT_FIELDS):raise PlanError('Missing/unknown selected source field')
    if r['study_ref'] not in r['provenance_sources'] or r['device_ref'] not in r['provenance_sources']:
      raise PlanError('Observation Provenance is not source-linked to its study and device')
    if r['unit']!='mm3':raise PlanError('Non-mm3 quantity in this prespecified fixture')
    if not isinstance(r['provenance_sources'],list):raise PlanError('Malformed provenance list')
    try:
      value=json.loads(r['value_raw'])
    except (ValueError,TypeError) as e:raise PlanError('Non-numeric value_raw') from e
    if not isinstance(value,(int,float)) or isinstance(value,bool) or not math.isfinite(value):
      raise PlanError('Invalid numeric value')
    group={k:r[k] for k in GROUP_FIELDS}
    sid=r['study_ref'];person=r['person_ref']
    if sid in study_context and canonical(group)!=canonical(study_context[sid]):
      raise PlanError('CONFLICTING_STUDY_CONTEXT: study reference has incompatible imaging context')
    if sid in study_person and study_person[sid]!=person:
      raise PlanError('CROSS_PERSON_STUDY_CONTEXT')
    study_context[sid]=group;study_person[sid]=person
    for key,lookup in [('study_uid',study_uids),('series_uid',series_uids)]:
      uid=r[key]
      if not re.fullmatch(r'[0-9]+(?:\.[0-9]+)+',uid) or len(uid)>64:
        raise PlanError('Syntactically invalid synthetic UID: '+key)
      if uid in lookup and lookup[uid]!=sid:raise PlanError('Duplicate synthetic '+key+' across distinct studies')
      lookup[uid]=sid
  return len(study_context)
def hybrid_encode(source):
    assert_source_integrity(source)
    ss=sorted(source,key=lambda x:x['observation_id'])
    by_visit=defaultdict(list)
    for r in ss:by_visit[visit_key(r)].append(r)
    ambiguous=set();visit_groups={};study_groups={}
    for vk,rs in sorted(by_visit.items()):
       contexts={canonical({k:r[k] for k in GROUP_FIELDS}) for r in rs}
       if len(contexts)==1:
           visit_groups[vk]={k:copy.deepcopy(rs[0][k]) for k in GROUP_FIELDS}
       else:
           ambiguous.add(vk)
           for r in rs:
               study_groups[r['study_ref']]={k:copy.deepcopy(r[k]) for k in GROUP_FIELDS}
    anatomies=sorted({canonical([r['region'],r['side']]) for r in ss})
    traces=sorted({canonical([r['trace_state'],r['trace_rule']]) for r in ss})
    amap={k:f'A{i:03d}' for i,k in enumerate(anatomies)}
    tmap={k:f'T{i:03d}' for i,k in enumerate(traces)}
    units={r['unit'] for r in ss};global_unit=next(iter(units)) if len(units)==1 else None
    payload={'format':PH35_FORMAT,'plan':HYBRID,'row_context':{},
             'group_context':{k:visit_groups[k] for k in sorted(visit_groups)},
             'study_context':{k:study_groups[k] for k in sorted(study_groups)},
             'ambiguous_visits':sorted(ambiguous),
             'codebooks':{'anatomy':{amap[x]:json.loads(x) for x in anatomies},
                          'trace':{tmap[x]:json.loads(x) for x in traces},
                          'global_unit':global_unit}}
    for r in ss:
        c={k:copy.deepcopy(r[k]) for k in LOCAL_FIELDS if k not in ['region','side','trace_state','trace_rule']}
        c['anatomy_code']=amap[canonical([r['region'],r['side']])]
        c['trace_code']=tmap[canonical([r['trace_state'],r['trace_rule']])]
        if global_unit is not None:c.pop('unit',None)
        if visit_key(r) in ambiguous:c['study_ref']=r['study_ref']
        payload['row_context'][r['observation_id']]=c
    return payload

def hybrid_decode(base,carrier):
    if carrier.get('format')!=PH35_FORMAT or carrier.get('plan')!=HYBRID:
       raise PlanError('Wrong hybrid format')
    rows=carrier['row_context'];vg=carrier['group_context'];sg=carrier['study_context']
    ambiguous=set(carrier['ambiguous_visits']);codes=carrier['codebooks']
    if set(rows)!={r['observation_id'] for r in base}:raise PlanError('Missing or extra hybrid row entry')
    if set(vg)&ambiguous:raise PlanError('Ambiguous visit stored as unique visit group')
    out=[]
    for b in base:
       rc=copy.deepcopy(rows[b['observation_id']]);vk=visit_key(b)
       if vk in ambiguous:
           try:group=sg[rc.pop('study_ref')]
           except (KeyError,TypeError) as e:raise PlanError('Invalid ambiguous study reference') from e
           if group['study_ref'] not in sg:pass  # only existence of the original key is relevant
       else:
           try:group=vg[vk]
           except KeyError as e:raise PlanError('Missing unique visit group') from e
       row=copy.deepcopy(b)
       try:
          row['region'],row['side']=codes['anatomy'][rc.pop('anatomy_code')]
          row['trace_state'],row['trace_rule']=codes['trace'][rc.pop('trace_code')]
          if codes['global_unit'] is not None:row['unit']=codes['global_unit']
       except (KeyError,IndexError,TypeError) as e:raise PlanError('Invalid hybrid codebook') from e
       row.update(group);row.update(rc)
       if sorted(row)!=sorted(BASE_FIELDS+CONTEXT_FIELDS):raise PlanError('Unreconstructed hybrid field')
       out.append(row)
    return sorted(out,key=lambda x:x['observation_id'])

def encode(source,plan):
  return hybrid_encode(source) if plan==HYBRID else make_carrier(source,plan)
def decode(base,carrier):
  return hybrid_decode(base,carrier) if carrier.get('plan')==HYBRID else reconstruct(base,carrier)
def exact_test(source,carrier):
  restored=decode(base_table(source),carrier)
  if canonical(restored)!=canonical(sorted(source,key=lambda r:r['observation_id'])):
     raise PlanError('Exact selected-field source-to-restored roundtrip failed')
  assert_source_integrity(restored)
  return restored
print('Six candidate codecs frozen for this notebook; no challenge labels loaded.')

Six candidate codecs frozen for this notebook; no challenge labels loaded.


## 3 · Frozen, person-disjoint experiment and source-proxy semantics
The participant split is determined by the SHA256 rank of each **synthetic patient ID**, not case labels or cohort membership. The original five codecs and sixth hybrid are evaluated on the training portion; the minimum-byte **admissible** plan is chosen using training metadata alone. Evaluate that frozen choice on the other nine people and report the separately computed test-set oracle as an *upper reference*, not as a deployable preselected method. Dictionary construction on new source records is part of encoding, not supervised learning.

In [6]:
people=sorted({r['person_ref'] for r in SOURCE})
assert len(people)==19
ORDER=sorted(people,key=lambda p:sha_bytes(('ICHI2027_P35_SPLIT_V1|'+p).encode('utf-8')))
TRAIN_PEOPLE=set(ORDER[:10]);TEST_PEOPLE=set(ORDER[10:]);assert len(TRAIN_PEOPLE)==10 and len(TEST_PEOPLE)==9 and not TRAIN_PEOPLE&TEST_PEOPLE
TRAIN=[r for r in SOURCE if r['person_ref'] in TRAIN_PEOPLE]
HOLDOUT=[r for r in SOURCE if r['person_ref'] in TEST_PEOPLE]
assert len(TRAIN)+len(HOLDOUT)==148
SPLIT=[{'synthetic_person_ref':p,'partition':'TRAIN' if p in TRAIN_PEOPLE else 'HOLDOUT',
        'split_rank':i+1} for i,p in enumerate(ORDER)]

def plan_rows(source,scenario):
    assert_source_integrity(source)
    results=[];carriers={}
    for plan in CANDIDATES:
       try:
          c=encode(source,plan);rec=exact_test(source,c)
          carriers[plan]=c
          results.append({'scenario':scenario,'plan':plan,'admissible':True,
                 'selected_field_roundtrip':len(rec),'metadata_bytes':byte_length(c),
                 'metadata_bytes_per_observation':round(byte_length(c)/len(source),3),
                 'visit_group_count':len(c.get('group_context',{})),
                 'fallback_study_group_count':len(c.get('study_context',{})),
                 'error':''})
       except (PlanError,KeyError,ValueError,TypeError) as e:
          results.append({'scenario':scenario,'plan':plan,'admissible':False,
                 'selected_field_roundtrip':0,'metadata_bytes':'',
                 'metadata_bytes_per_observation':'',
                 'visit_group_count':'','fallback_study_group_count':'','error':str(e)[:260]})
    return results,carriers

TRAIN_ROWS,TRAIN_CARRIERS=plan_rows(TRAIN,'TRAIN_10_PERSONS')
assert 'INLINE' in TRAIN_CARRIERS and HYBRID in TRAIN_CARRIERS
FROZEN_PLAN=min(TRAIN_CARRIERS,key=lambda p:(byte_length(TRAIN_CARRIERS[p]),p))
TEST_ROWS,TEST_CARRIERS=plan_rows(HOLDOUT,'HELDOUT_9_PERSONS')
assert FROZEN_PLAN in TEST_CARRIERS, 'Selected train plan is inadmissible on natural holdout; record as transfer failure'
TEST_ORACLE=min(TEST_CARRIERS,key=lambda p:(byte_length(TEST_CARRIERS[p]),p))
TRAIN_INLINE=byte_length(TRAIN_CARRIERS['INLINE'])
TEST_INLINE=byte_length(TEST_CARRIERS['INLINE'])
TEST_FROZEN=byte_length(TEST_CARRIERS[FROZEN_PLAN])
TEST_ORACLE_BYTES=byte_length(TEST_CARRIERS[TEST_ORACLE])
PLAN_AUDIT=TRAIN_ROWS+TEST_ROWS
print('Train people/observations:',len(TRAIN_PEOPLE),len(TRAIN),'frozen plan:',FROZEN_PLAN)
print('Holdout people/observations:',len(TEST_PEOPLE),len(HOLDOUT),'test metadata bytes:',TEST_FROZEN,'vs inline:',TEST_INLINE)
print('Holdout post hoc oracle:',TEST_ORACLE,TEST_ORACLE_BYTES)

Train people/observations: 10 79 frozen plan: VISIT_CODED
Holdout people/observations: 9 69 test metadata bytes: 26546 vs inline: 43157
Holdout post hoc oracle: VISIT_CODED 26546


## 4 · Prespecified structural stress tests
The following data are **deterministic modifications of the 9 synthetic holdout persons**—not new external held-out participants. Scenario A adds one different MRI study to an existing person/visit. Scenario B removes one 12-month left hippocampal record. Scenario C introduces an extra anatomy code absent from training. Scenario D adds multiple second-study visits. Finally, a contradictory synthetic study UID is injected and the source must be **quarantined** before any encoder can report successful compression. Preexisting FHIR and OMOP files are not modified.

In [7]:
STRESSES={}
def synthetic_duplicate(source,idx=0,tag='SECOND_STUDY',extra_region=None):
    s=copy.deepcopy(sorted(source,key=lambda r:r['observation_id']))
    original=s[idx];new=copy.deepcopy(original)
    new['observation_id']=sha_bytes(('P35|'+tag+'|'+original['observation_id']).encode('utf-8'))[:24]
    new['source_observation_ref']='Observation/p35-'+tag.lower()+'-'+new['observation_id']
    new['provenance_id']='Provenance/p35-'+tag.lower()+'-'+new['observation_id']
    if extra_region is not None:
       new['region']=extra_region
    else:
       prefix=int(sha_bytes(new['observation_id'].encode('utf-8'))[:10],16)
       new['study_ref']='ImagingStudy/P35-'+tag+'-'+new['observation_id']
       new['study_uid']='2.25.'+str(10**37+prefix)
       new['series_uid']='2.25.'+str(20**37+prefix)
       new['provenance_sources']=[new['study_ref'],new['device_ref']]
    s.append(new)
    assert_source_integrity(s)
    return s,new

STRESSES['HOLDOUT_CLEAN']=HOLDOUT
mixed,mixnew=synthetic_duplicate(HOLDOUT,tag='MIXED_VISIT_1')
STRESSES['MIXED_STUDY_SAME_VISIT']=mixed
sparse=copy.deepcopy(HOLDOUT)
sparse_index=next(i for i,r in enumerate(sparse) if r['visit']=='12-month' and r['region']=='hippocampus' and r['side']=='left')
dropped=sparse.pop(sparse_index)
assert_source_integrity(sparse)
STRESSES['MISSING_FOLLOWUP_HEMISPHERE']=sparse
novel,_=synthetic_duplicate(HOLDOUT,tag='UNSEEN_ANATOMY',extra_region='amygdala')
STRESSES['UNSEEN_ANATOMY_CODE']=novel
multi=copy.deepcopy(HOLDOUT)
# Four distinct original person/visit pairs; deterministic, no source-label inspection.
uniq=sorted({(r['person_ref'],r['visit']) for r in multi})[:4]
for i,(person,visit) in enumerate(uniq):
    pos=next(j for j,r in enumerate(multi) if (r['person_ref'],r['visit'])==(person,visit))
    multi,_=synthetic_duplicate(multi,idx=pos,tag='MULTI_'+str(i+1))
STRESSES['FOUR_MIXED_VISITS']=multi

SCENARIO_RESULTS=[];SCENARIO_PLAN_ROWS=[];SCENARIO_CARRIERS={};POLICY_ROWS=[]
def source_policy(rows):
    by=defaultdict(list)
    for r in rows:by[r['person_ref']].append(r)
    out={}
    for p,events in by.items():
       matches=[];qualified=True
       for visit in ('baseline','12-month'):
          for side in ('left','right'):
             actual=[r for r in events if r['region']=='hippocampus' and r['side']==side and r['visit']==visit]
             matches.append(len(actual)==1)
             for r in actual:
                qualified&=(r['trace_state']=='PASS' and r['unit']=='mm3' and
                       r['study_ref'] in r['provenance_sources'] and r['device_ref'] in r['provenance_sources'])
       out[p]={'source_proxy':all(matches),'evidence_qualified_proxy':all(matches) and bool(qualified)}
    return out

for name,subset in STRESSES.items():
    rows,carriers=plan_rows(subset,name)
    SCENARIO_PLAN_ROWS+=rows;SCENARIO_CARRIERS[name]=carriers
    inline=byte_length(carriers['INLINE'])
    posthoc=min(carriers,key=lambda x:(byte_length(carriers[x]),x))
    fallback='NONE'
    if FROZEN_PLAN in carriers:chosen=FROZEN_PLAN
    elif HYBRID in carriers:
       chosen=HYBRID;fallback='HYBRID_CODED_AFTER_FROZEN_PLAN_INADMISSIBLE'
    elif 'STUDY_CODED' in carriers:
       chosen='STUDY_CODED';fallback='STUDY_CODED_AFTER_FROZEN_PLAN_INADMISSIBLE'
    else:
       chosen='INLINE';fallback='INLINE_AFTER_FROZEN_PLAN_INADMISSIBLE'
    recovered=exact_test(subset,carriers[chosen]);src_policy=source_policy(subset)
    rec_policy=source_policy(recovered)
    assert src_policy==rec_policy
    for p in sorted(src_policy):
       POLICY_ROWS.append({'scenario':name,'synthetic_person_ref':p,
          'source_proxy_member':src_policy[p]['source_proxy'],
          'restored_proxy_member':rec_policy[p]['source_proxy'],
          'source_evidence_qualified_member':src_policy[p]['evidence_qualified_proxy'],
          'restored_evidence_qualified_member':rec_policy[p]['evidence_qualified_proxy'],
          'source_restored_exact_agreement':True,'official_target_cohort_equivalence':False,'MCI_status':'NOT_ATTESTED'})
    SCENARIO_RESULTS.append({'scenario':name,'synthetic_people':len(src_policy),'synthetic_observations':len(subset),
         'frozen_train_plan':FROZEN_PLAN,'frozen_plan_admissible':FROZEN_PLAN in carriers,
         'fallback_status':fallback,'deployed_plan':chosen,'deployed_metadata_bytes':byte_length(carriers[chosen]),
         'inline_metadata_bytes':inline,
         'deployed_saving_vs_inline_percent':round(100*(inline-byte_length(carriers[chosen]))/inline,4),
         'posthoc_oracle_plan':posthoc,'posthoc_oracle_metadata_bytes':byte_length(carriers[posthoc]),
         'hybrid_admissible':HYBRID in carriers,
         'hybrid_metadata_bytes':byte_length(carriers[HYBRID]) if HYBRID in carriers else '',
         'study_coded_metadata_bytes':byte_length(carriers['STUDY_CODED']) if 'STUDY_CODED' in carriers else '',
         'visit_coded_admissible':'VISIT_CODED' in carriers,
         'source_vs_restored_proxy_person_agreement':sum(src_policy[p]==rec_policy[p] for p in src_policy)})

# Source-level contradiction MUST be blocked before assessing any plan, including INLINE.
conflict=copy.deepcopy(HOLDOUT)
other=next(i for i,r in enumerate(conflict) if r['study_ref']==conflict[0]['study_ref'] and r['observation_id']!=conflict[0]['observation_id'])
conflict[other]['study_uid']='2.25.999999999999999999999999999999'
quarantine_error=''
try:assert_source_integrity(conflict)
except PlanError as ex:quarantine_error=str(ex)
assert quarantine_error.startswith('CONFLICTING_STUDY_CONTEXT:')
SCENARIO_RESULTS.append({'scenario':'CONFLICTING_STUDY_UID_QUARANTINE', 'synthetic_people':len(TEST_PEOPLE),
 'synthetic_observations':len(conflict),'frozen_train_plan':FROZEN_PLAN,'frozen_plan_admissible':False,
 'fallback_status':'QUARANTINED_SOURCE_NOT_ENCODED','deployed_plan':'NONE','deployed_metadata_bytes':'',
 'inline_metadata_bytes':'','deployed_saving_vs_inline_percent':'','posthoc_oracle_plan':'NONE',
 'posthoc_oracle_metadata_bytes':'','hybrid_admissible':False,'hybrid_metadata_bytes':'',
 'study_coded_metadata_bytes':'','visit_coded_admissible':False,
 'source_vs_restored_proxy_person_agreement':'NOT_TESTED_SOURCE_INVALID'})
assert 'VISIT_CODED' not in SCENARIO_CARRIERS['MIXED_STUDY_SAME_VISIT']
assert HYBRID in SCENARIO_CARRIERS['MIXED_STUDY_SAME_VISIT']
assert 'VISIT_CODED' not in SCENARIO_CARRIERS['FOUR_MIXED_VISITS']
assert HYBRID in SCENARIO_CARRIERS['FOUR_MIXED_VISITS']
print('Frozen plan:',FROZEN_PLAN,'scenarios:',[(r['scenario'],r['deployed_plan']) for r in SCENARIO_RESULTS])

Frozen plan: VISIT_CODED scenarios: [('HOLDOUT_CLEAN', 'VISIT_CODED'), ('MIXED_STUDY_SAME_VISIT', 'HYBRID_CODED'), ('MISSING_FOLLOWUP_HEMISPHERE', 'VISIT_CODED'), ('UNSEEN_ANATOMY_CODE', 'VISIT_CODED'), ('FOUR_MIXED_VISITS', 'HYBRID_CODED'), ('CONFLICTING_STUDY_UID_QUARANTINE', 'NONE')]


## 5 · Integrity, invariance, and false-completion controls
Source comparison detects tampering in the same synthetic test; a raw checksum does **not** authenticate a maliciously replaced source-plus-manifest pair. This notebook also guards against a misleading percentage: adding source field dictionaries, carrier envelope, and grouping identifiers must be counted in all reported byte totals. The `HOLDOUT` savings are a person-disjoint synthetic result only.

In [8]:
TESTS=[]
def check(name,fn):
 try:fn();TESTS.append({'test':name,'passed':True,'result':'PASS'})
 except Exception as ex:TESTS.append({'test':name,'passed':False,'result':repr(ex)})
def t34():
  assert len(R34)==148 and byte_length(C34)==56408
  assert canonical(R34)==canonical(sorted(SOURCE,key=lambda x:x['observation_id']))
def t_disjoint():
  assert not TRAIN_PEOPLE&TEST_PEOPLE and len(TRAIN_PEOPLE)==10 and len(TEST_PEOPLE)==9
  assert not any('case_label' in r or 'injected' in r for r in SOURCE)
def t_train_fixed():
  assert FROZEN_PLAN==min(TRAIN_CARRIERS,key=lambda p:(byte_length(TRAIN_CARRIERS[p]),p))
  assert FROZEN_PLAN in TEST_CARRIERS
def t_holdout():
  for plan,c in TEST_CARRIERS.items():assert len(exact_test(HOLDOUT,c))==len(HOLDOUT)
def t_mixed():
  assert 'VISIT_CODED' not in SCENARIO_CARRIERS['MIXED_STUDY_SAME_VISIT']
  assert len(exact_test(mixed,SCENARIO_CARRIERS['MIXED_STUDY_SAME_VISIT'][HYBRID]))==len(mixed)
def t_multi():
  assert 'VISIT_CODED' not in SCENARIO_CARRIERS['FOUR_MIXED_VISITS']
  assert HYBRID in SCENARIO_CARRIERS['FOUR_MIXED_VISITS']
  assert len(exact_test(multi,SCENARIO_CARRIERS['FOUR_MIXED_VISITS'][HYBRID]))==len(multi)
def t_sparse():
  before=source_policy(HOLDOUT);after=source_policy(sparse)
  assert any(before[p]!=after[p] for p in before), 'Sparse control did not alter test proxy membership'
  assert source_policy(exact_test(sparse,SCENARIO_CARRIERS['MISSING_FOLLOWUP_HEMISPHERE'][HYBRID]))==after
def t_unseen():
  assert ('amygdala','left') not in {(r['region'],r['side']) for r in TRAIN}
  h=SCENARIO_CARRIERS['UNSEEN_ANATOMY_CODE'][HYBRID]
  assert ['amygdala','left'] in list(h['codebooks']['anatomy'].values())
  assert len(exact_test(novel,h))==len(novel)
def t_quarantine():
  try:assert_source_integrity(conflict)
  except PlanError as e:assert 'CONFLICTING_STUDY_CONTEXT' in str(e)
  else:raise AssertionError('Conflicting source passed integrity gate')
def t_missing():
  bad=copy.deepcopy(SCENARIO_CARRIERS['MIXED_STUDY_SAME_VISIT'][HYBRID]);del bad['row_context'][mixnew['observation_id']]
  try:exact_test(mixed,bad)
  except PlanError:pass
  else:raise AssertionError('Missing hybrid sidecar silently accepted')
def t_tamper():
  bad=copy.deepcopy(SCENARIO_CARRIERS['HOLDOUT_CLEAN'][HYBRID]);key=next(iter(bad['group_context']))
  bad['group_context'][key]['study_uid']='2.25.123456'
  try:exact_test(HOLDOUT,bad)
  except PlanError:pass
  else:raise AssertionError('Tampering passed source-anchored comparison')
def t_determinism():
  assert canonical(hybrid_encode(HOLDOUT))==canonical(hybrid_encode(list(reversed(HOLDOUT))))
  assert all(r['metadata_bytes']==byte_length(SCENARIO_CARRIERS['HOLDOUT_CLEAN'][r['plan']]) for r in SCENARIO_PLAN_ROWS if r['scenario']=='HOLDOUT_CLEAN' and r['admissible'])
def t_query():
  assert all(x['source_proxy_member']==x['restored_proxy_member'] and x['source_evidence_qualified_member']==x['restored_evidence_qualified_member'] for x in POLICY_ROWS)
def t_original_gates():
  assert P34['unresolved']['original_vocabulary_gaps_blocked']==36 and not P34['unresolved']['official_MiCDM_loading']

for name,fn in [
 ('phase34_selected_field_full_reproduction',t34),('person_disjoint_no_label_input',t_disjoint),
 ('train_only_plan_frozen',t_train_fixed),('all_valid_holdout_plan_roundtrips',t_holdout),
 ('two_studies_same_visit_hybrid_fallback',t_mixed),('four_mixed_visits_hybrid_reconstruction',t_multi),
 ('deliberate_missing_followup_policy_preserved',t_sparse),('unseen_anatomy_dictionary_lossless',t_unseen),
 ('source_conflicting_study_quarantined',t_quarantine),('hybrid_missing_row_context_rejected',t_missing),
 ('source_anchored_study_tampering_detected',t_tamper),('byte_count_and_replay_deterministic',t_determinism),
 ('source_restored_proxy_queries_preserved',t_query),('standards_and_vocabulary_gates_unmodified',t_original_gates)]:
 check(name,fn)
if not all(x['passed'] for x in TESTS):raise RuntimeError('Phase35 synthetic tests failed: '+str(TESTS))
print('PASS:',len(TESTS),'/',len(TESTS),'synthetic experiment checks.')

PASS: 14 / 14 synthetic experiment checks.


## 6 · Save auditable outputs in the Phase 34 Google Drive folder
The study's **predeclared split, all 6 encoder results per scenario, a source/target-only proxy agreement table, failures and input hashes** are saved. A small, human-readable method specification accompanies the numeric results. No private files or named Athena concept IDs are exported.

In [9]:
OUT={
 'shareable':OUT_DIR/'ICHI2027_Phase35_SHAREABLE.json',
 'split':OUT_DIR/'ICHI2027_Phase35_PERSON_DISJOINT_SPLIT.csv',
 'plans':OUT_DIR/'ICHI2027_Phase35_PLAN_SPLIT_EVALUATION.csv',
 'scenario_plans':OUT_DIR/'ICHI2027_Phase35_SCENARIO_PLAN_AUDIT.csv',
 'scenarios':OUT_DIR/'ICHI2027_Phase35_STRESS_SCENARIOS.csv',
 'tests':OUT_DIR/'ICHI2027_Phase35_RECONSTRUCTION_TESTS.csv',
 'policy':OUT_DIR/'ICHI2027_Phase35_SYNTHETIC_PROXY_PRESERVATION.csv',
 'method':OUT_DIR/'ICHI2027_Phase35_METHOD_SPECIFICATION.json',
}
def write_csv(path,rows):
 assert rows
 with path.open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys()));w.writeheader()
    for r in rows:w.writerow({k:(canonical(v) if isinstance(v,(list,dict)) else v) for k,v in r.items()})
for key,rows in [('split',SPLIT),('plans',PLAN_AUDIT),('scenario_plans',SCENARIO_PLAN_ROWS),
                 ('scenarios',SCENARIO_RESULTS),('tests',TESTS),('policy',POLICY_ROWS)]:
 write_csv(OUT[key],rows)
method={
 'experiment':'person-disjoint synthetic transport plan family transfer and mixed-visit group adaptation',
 'version':'P35_v1','prespecified_split':'SHA256 rank ICHI2027_P35_SPLIT_V1|person; first 10 TRAIN, remaining 9 HOLDOUT',
 'input_source':'exact SHA256-verified Phase34 selected-field synthetic canonical manifest',
 'plan_selection':'smallest canonical JSON bytes among six structurally admissible exact-roundtrip plans on TRAIN only',
 'candidate_plans':CANDIDATES,
 'hybrid':'VISIT group only if context is unique within person/visit; ambiguous visits use STUDY context, affected rows carry study_ref; repeated anatomy and TRACE fields use deterministic per-slice dictionaries',
 'transfer':'frozen plan family applied on person-disjoint HOLDOUT; codebooks are learned unsupervised from the to-be-encoded partition; separately report test-set oracle post hoc',
 'structural_fallback':'if train plan inadmissible on shifted source, use HYBRID_CODED if source passes gate and all selected fields reconstruct; otherwise STUDY_CODED then INLINE',
 'metrics':['full carrier canonical metadata bytes INCLUDING dictionaries and headers','bytes per observation','source-restored exact selected-field reconstruction','source-restored synthetic source-proxy person agreement','plan admissibility under multi-study visits'],
 'limitations':['synthetic controls derived from same developer as pipeline','not independently authored or externally blinded','not a comparison to standards-compliant Arm A or official MI-CDM','no real MRI, no MCI clinical membership','original Arm C not executed','novelty relative to compression and database normalization literature unverified'],
 'blocked_gates':{'frozen_athena_provided':False,'original_vocab_gaps':36,'faithful_armA':False,'original_armC':False,'official_MICDM_load':False,'real_MRI':False,'four_arm_comparison':False}
}
OUT['method'].write_text(json.dumps(method,indent=2,ensure_ascii=False),encoding='utf-8')
HOLDOUT_SAVINGS=round(100*(TEST_INLINE-TEST_FROZEN)/TEST_INLINE,4)
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'35_person_disjoint_synthetic_transport_generalization_and_mixed_study_hybrid',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Synthetic-only; no ADNI/clinical records, real DICOM identifiers, private Athena concepts or private Drive paths',
 'upstream':{'phase34_shareable_sha256':sha(P34_PATH),
             'phase34_source_sha256':sha(P34_FILES['source']),
             'phase34_base_sha256':sha(P34_FILES['base']),
             'phase34_carrier_sha256':sha(P34_FILES['carrier']),
             'imagewg_commit':P34['upstream']['pinned_imagewg_commit'],
             'omop55_commit':P34['upstream']['pinned_omop55_commit']},
 'phase34_reproduced':{'full_cohort_people':19,'full_cohort_observations':148,
                       'selected_plan':P34['method']['selected_plan'],
                       'metadata_byte_reduction_vs_inline_percent':P34['method']['metadata_byte_reduction_vs_inline_percent'],
                       'selected_field_roundtrip_verified':True},
 'person_disjoint_synthetic_transfer':{'train_people':10,'holdout_people':9,'train_observations':len(TRAIN),
                 'holdout_observations':len(HOLDOUT),'selected_train_plan':FROZEN_PLAN,
                 'holdout_frozen_plan_admissible':True,'holdout_inline_metadata_bytes':TEST_INLINE,
                 'holdout_frozen_plan_metadata_bytes':TEST_FROZEN,
                 'holdout_reduction_vs_inline_percent':HOLDOUT_SAVINGS,
                 'holdout_posthoc_oracle_plan':TEST_ORACLE,
                 'holdout_posthoc_oracle_metadata_bytes':TEST_ORACLE_BYTES,
                 'externally_independent_holdout':False,'outcome_labels_used_to_select_plan':False},
 'mixed_study':{'two_studies_same_visit_phase34_visit_plan_admissible':False,
                'two_studies_same_visit_hybrid_admissible':True,
                'multi_visit_stress_hybrid_admissible':True,
                'conflicting_source_study_uid_quarantined':True},
 'tests':{'synthetic_checks':len(TESTS),'checks_passing':sum(x['passed'] for x in TESTS),
          'synthetic_scenarios_including_quarantine':len(SCENARIO_RESULTS),
          'source_restored_synthetic_proxy_agreements_all_scenarios':all(x['source_restored_exact_agreement'] for x in POLICY_ROWS)},
 'gates':{'frozen_private_athena_snapshot_supplied':False,'original_vocab_gaps_blocked':36,
          'source_supported_19_bindings_approved':False,'synthetic_faithful_armA_completed':False,
          'original_generic_armC_executed':False,'official_MICDM_load_completed':False,
          'real_MRI_rows_processed':0,'true_four_arm_comparison_completed':False,
          'OMOP_target_cohort_equivalence_tested':False,'novelty_or_clinical_effect_established':False},
 'artifact_sha256':{k:sha(p) for k,p in OUT.items() if k!='shareable'},
 'next_stage_gate':'Acquire authorized frozen Athena CONCEPT.csv and source-supported review for 19 bindings; run valid Arm A, independent non-developer evaluation and eligible real MRI before publication-level claims.'
}
OUT['shareable'].write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print('Phase35 result:',FROZEN_PLAN,'holdout reduction vs inline',HOLDOUT_SAVINGS,'percent')
for key,p in OUT.items():print('SAVED',p.name)

Phase35 result: VISIT_CODED holdout reduction vs inline 38.4897 percent
SAVED ICHI2027_Phase35_SHAREABLE.json
SAVED ICHI2027_Phase35_PERSON_DISJOINT_SPLIT.csv
SAVED ICHI2027_Phase35_PLAN_SPLIT_EVALUATION.csv
SAVED ICHI2027_Phase35_SCENARIO_PLAN_AUDIT.csv
SAVED ICHI2027_Phase35_STRESS_SCENARIOS.csv
SAVED ICHI2027_Phase35_RECONSTRUCTION_TESTS.csv
SAVED ICHI2027_Phase35_SYNTHETIC_PROXY_PRESERVATION.csv
SAVED ICHI2027_Phase35_METHOD_SPECIFICATION.json


In [10]:
# Drive readback: assert every output matches the SHAREABLE manifest.
readback=json.loads(OUT['shareable'].read_text(encoding='utf-8'))
for key,digest in readback['artifact_sha256'].items():
  assert sha(OUT[key])==digest, ('Phase35 readback hash mismatch',key)
assert len(load_csv(OUT['split']))==19
assert len(load_csv(OUT['plans']))==12
assert len(load_csv(OUT['tests']))==14
assert all(r['passed']=='True' for r in load_csv(OUT['tests']))
assert all(r['source_restored_exact_agreement']=='True' for r in load_csv(OUT['policy']))
assert readback['gates']['original_vocab_gaps_blocked']==36
print('PHASE 35 SYNTHETIC GENERALIZATION EXPERIMENT COMPLETE — Drive readback and SHA gates passed.')
print('Please upload SHAREABLE, PLAN_SPLIT_EVALUATION, STRESS_SCENARIOS and RECONSTRUCTION_TESTS.')

PHASE 35 SYNTHETIC GENERALIZATION EXPERIMENT COMPLETE — Drive readback and SHA gates passed.
Please upload SHAREABLE, PLAN_SPLIT_EVALUATION, STRESS_SCENARIOS and RECONSTRUCTION_TESTS.


### After running
Upload these four **public synthetic-only** files: `ICHI2027_Phase35_SHAREABLE.json`, `ICHI2027_Phase35_PLAN_SPLIT_EVALUATION.csv`, `ICHI2027_Phase35_STRESS_SCENARIOS.csv`, and `ICHI2027_Phase35_RECONSTRUCTION_TESTS.csv`. Retain the additional scenario-plan audit, method specification, synthetic person split and proxy-preservation files on Drive for reproducibility.

**Do not use this output to claim independent clinical generalization, official OMOP/MI-CDM cohort equivalence, novelty, real-DICOM provenance or resolution of the 36 blocked vocabulary instances.**